In [32]:
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
model = SentenceTransformer("anass1209/resume-job-matcher-all-MiniLM-L6-v2")
# Run inference
sentences = [
    'Developed and maintained core backend services using Python and Django, focusing on scalability and efficiency. Implemented RESTful APIs for data retrieval and manipulation.  Worked extensively with PostgreSQL for data storage and retrieval.  Responsible for optimizing database queries and improving API response times.  Experience with model fine-tuning for semantic search and document retrieval using pre-trained embedding models like Sentence Transformers or similar libraries, specifically for improving the relevance of search results and document matching within the web application.  Experience using vector databases (e.g., ChromaDB, Weaviate) preferred.',
    '## Senior Backend Engineer\n\n*   **ABC Corp** | 2020 - Present\n*   Led development of a new REST API for user authentication and profile management using Python and Django.\n*   Managed a PostgreSQL database, optimizing queries and schema design for improved performance, resulting in a 20% reduction in average API response time.\n*   Improved system scalability through efficient code design and load balancing techniques.\n*   Experience using pre-trained embedding models (BERT) for natural language processing tasks to improve search accuracy, with focus on keyphrase extraction and content similarity comparison for the recommendations engine. Proficient in Flask.',
    "PhD in Computer Science, University of California, Berkeley (2018-2023). Dissertation: 'Adversarial Robustness in NLP for Cybersecurity Applications.' Focused on fine-tuning BERT for malware detection and social engineering attacks. Proficient in Python, TensorFlow, and AWS. Published in top-tier NLP and security conferences. Experienced with large datasets and model evaluation metrics.\n\nMaster of Science in Cybersecurity, Johns Hopkins University (2016-2018). Relevant coursework included Machine Learning, Data Mining, and Network Security. Developed a system for anomaly detection using a recurrent neural network (RNN). Familiar with Python and cloud computing platforms. Good understanding of NLP concepts, but limited experience fine-tuning transformer models. Strong understanding of Information Security Principles.\n\nBachelor of Science in Computer Engineering, Carnegie Mellon University (2012-2016). Relevant coursework: Artificial Intelligence, Database Management, and Software Engineering. Project experience: Developed a web application using Python. No direct experience with fine-tuning NLP models, but a strong foundation in programming and data structures.  Familiar with cloud infrastructure concepts. Possess CISSP certification.",
]
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# Get the similarity scores for the embeddings
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

(3, 384)
torch.Size([3, 3])


In [7]:
from sentence_transformers import SentenceTransformer, util
import fitz  # PyMuPDF for reading PDFs
import glob

# ---------- Step 1: Load model ----------
model = SentenceTransformer("anass1209/resume-job-matcher-all-MiniLM-L6-v2")

# ---------- Step 2: Function to extract text from PDF ----------
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text.strip()

# ---------- Step 3: Load job description ----------
job_description_path = "job_description.pdf"  # Change to your JD PDF path
job_text = extract_text_from_pdf("/media/ishan/9c63e104-0a83-464e-8d4f-7e0675ab17ef/Sports Coach/ml_job_description.pdf")

# ---------- Step 4: Load all resumes ----------
resume_folder = "Random_Resume/*.pdf"  # Change to your resumes folder path
resume_files = glob.glob(resume_folder)

resumes = []
for file in resume_files:
    resumes.append({
        "file": file,
        "text": extract_text_from_pdf(file)
    })

# ---------- Step 5: Embed job description & resumes ----------
job_embedding = model.encode(job_text, convert_to_tensor=True)
resume_embeddings = [model.encode(r["text"], convert_to_tensor=True) for r in resumes]

# ---------- Step 6: Compute similarities ----------
similarities = [util.cos_sim(job_embedding, emb).item() for emb in resume_embeddings]

# ---------- Step 7: Rank resumes by similarity ----------
ranked_resumes = sorted(
    zip(resume_files, similarities),
    key=lambda x: x[1],
    reverse=True
)

# ---------- Step 8: Print top 3 ----------
print("Top matching resumes for the job description:\n")
for i, (file, score) in enumerate(ranked_resumes, start=1):
    print(f"{i}. {file} — Similarity: {score:.4f}")


Top matching resumes for the job description:

1. Random_Resume/10.pdf — Similarity: 0.8899
2. Random_Resume/3.pdf — Similarity: 0.8634
3. Random_Resume/7.pdf — Similarity: 0.8633
4. Random_Resume/6.pdf — Similarity: 0.8504
5. Random_Resume/11.pdf — Similarity: 0.8488
6. Random_Resume/2.pdf — Similarity: 0.8459
7. Random_Resume/12.pdf — Similarity: 0.8429
8. Random_Resume/9.pdf — Similarity: 0.8428
9. Random_Resume/4.pdf — Similarity: 0.8402
10. Random_Resume/1.pdf — Similarity: 0.8342
11. Random_Resume/5.pdf — Similarity: 0.8263
12. Random_Resume/8.pdf — Similarity: 0.7879


In [6]:
import fitz  # PyMuPDF for PDF text extraction
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
import torch
import torch.nn.functional as F
import glob

# ---------- Load model ----------
base_model = AutoModel.from_pretrained("BAAI/bge-large-en-v1.5")
model = PeftModel.from_pretrained(base_model, "shashu2325/resume-job-matcher-lora")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")

# ---------- Function: Extract text from PDF ----------
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text.strip()

# ---------- Function: Get normalized embedding ----------
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        padding="max_length",
        truncation=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
        emb = F.normalize(emb, p=2, dim=1)
    return emb

# ---------- Load Job Description ----------
job_pdf_path = "ml_job_description.pdf"  # Change to your JD file path
job_text = extract_text_from_pdf(job_pdf_path)
job_emb = get_embedding(job_text)

# ---------- Load Resumes ----------
resume_folder = "Random_Resume/*.pdf"  # Change path to your resumes folder
resume_files = glob.glob(resume_folder)

results = []

for resume_file in resume_files:
    resume_text = extract_text_from_pdf(resume_file)
    resume_emb = get_embedding(resume_text)

    # Cosine similarity
    similarity = torch.sum(resume_emb * job_emb, dim=1)
    match_score = torch.sigmoid(similarity).item()

    results.append((resume_file, match_score))

# ---------- Rank and Display ----------
top_matches = sorted(results, key=lambda x: x[1], reverse=True)

print("\nTop Matching Resumes:\n")
for i, (file, score) in enumerate(top_matches, start=1):
    print(f"{i}. {file} — Match Score: {score:.4f}")


Top Matching Resumes:

1. Random_Resume/2.pdf — Match Score: 0.6680
2. Random_Resume/6.pdf — Match Score: 0.6642
3. Random_Resume/9.pdf — Match Score: 0.6587
4. Random_Resume/11.pdf — Match Score: 0.6521
5. Random_Resume/5.pdf — Match Score: 0.6474
6. Random_Resume/1.pdf — Match Score: 0.6455
7. Random_Resume/7.pdf — Match Score: 0.6379
8. Random_Resume/12.pdf — Match Score: 0.6360
9. Random_Resume/3.pdf — Match Score: 0.6335
10. Random_Resume/4.pdf — Match Score: 0.6322
11. Random_Resume/8.pdf — Match Score: 0.6291
12. Random_Resume/10.pdf — Match Score: 0.6262


In [2]:
top_matches

[('Random_Resume/2.pdf', 0.6680141687393188),
 ('Random_Resume/6.pdf', 0.6641677021980286),
 ('Random_Resume/9.pdf', 0.6586793661117554)]

In [ ]:
import fitz  # PyMuPDF
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig
import glob
import json

# ---------- Function: Extract text from PDF ----------
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text.strip()

# ---------- Load Model ----------
device = "cuda" if torch.cuda.is_available() else "cpu"

base_model_name = "akjindal53244/Llama-3.1-Storm-8B"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

peft_model_id = "LlamaFactoryAI/cv-job-description-matching"
PeftConfig.from_pretrained(peft_model_id)  # Just to load config
model = PeftModel.from_pretrained(base_model, peft_model_id)

# ---------- Load Job Description ----------
job_pdf_path = "ml_job_description.pdf"  # Change path
job_text = extract_text_from_pdf(job_pdf_path)

# ---------- Load Resumes ----------
resume_files = glob.glob("Random_Resume/*.pdf")  # Change path

results = []

for resume_file in resume_files:
    cv_text = extract_text_from_pdf(resume_file)

    # Build prompt
    messages = [
        {
            "role": "system",
            "content": """You are an advanced AI model designed to analyze the compatibility between a CV and a job description. You will receive a CV and a job description. Your task is to output a structured JSON format that includes:

1. matching_analysis: Analyze the CV against the job description to identify key strengths and gaps.
2. description: Summarize the relevance of the CV to the job description in a few concise sentences.
3. score: Provide a numerical compatibility score (0-100) based on qualifications, skills, and experience.
4. recommendation: Suggest actions for the candidate to improve their match or readiness for the role.

Your output must be in JSON format as follows:
{
  "matching_analysis": "Your detailed analysis here.",
  "description": "A brief summary here.",
  "score": 85,
  "recommendation": "Your suggestions here."
}
""",
        },
        {"role": "user", "content": f"<CV> {cv_text} </CV>\n<job_description> {job_text} </job_description>"},
    ]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    outputs = model.generate(inputs, max_new_tokens=512)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Attempt to parse JSON from model output
    try:
        result_json = json.loads(generated_text)
        score = result_json.get("score", 0)
    except json.JSONDecodeError:
        result_json = {"error": "Failed to parse JSON", "raw_output": generated_text}
        score = 0

    results.append({
        "file": resume_file,
        "analysis": result_json,
        "score": score
    })

# ---------- Sort by score & get top 3 ----------
top_3 = sorted(results, key=lambda x: x["score"], reverse=True)[:3]

# ---------- Print results ----------
print("\nTop 3 Matching Resumes:\n")
for i, res in enumerate(top_3, start=1):
    print(f"{i}. {res['file']} — Score: {res['score']}")
    print(f"   Description: {res['analysis'].get('description', 'N/A')}")
    print(f"   Recommendation: {res['analysis'].get('recommendation', 'N/A')}")
    print()


Fetching 4 files:   0%|          | 0/4 [04:31<?, ?it/s]


In [1]:
import google.generativeai as genai
import markdown
import random
from weasyprint import HTML

model = genai.GenerativeModel('gemini-1.5-flash')

generation_config = genai.GenerationConfig(
    temperature=0.7 # Increased temperature for more creative output
)

# Dictionaries of random data to choose from
first_names = ["Elias", "Seraphina", "Julian", "Aurora", "Finnian", "Maeve", "Caelum", "Zara", "Kaelen", "Lyra"]
last_names = ["Vance", "Sterling", "Kaelin", "Renwick", "Cromwell", "Pendelton", "Beaumont", "Stratton", "Thornton", "Huxley"]
job_titles = ["Senior AI Engineer", "Lead Machine Learning Scientist", "Principal Data Architect", "Machine Learning Research Fellow", "Computer Vision Specialist"]
companies = ["Innovate AI Labs", "Quantum Horizon Corp.", "Veridian Dynamics", "Aetherium Analytics", "Synthetica Global"]
universities = ["Imperial College London", "Carnegie Mellon University", "Stanford University", "Massachusetts Institute of Technology (MIT)", "University of Oxford"]
languages = ["Python", "C++", "Java", "R", "Go"]
frameworks = ["TensorFlow", "PyTorch", "Scikit-learn", "Hugging Face Transformers", "Keras"]
cloud_platforms = ["AWS", "Google Cloud Platform (GCP)", "Microsoft Azure"]

# The prompt is now an f-string that dynamically pulls from the dictionaries.
prompt_template = """
Generate a detailed long resume for a random person with a focus on AI/ML.
The candidate's name is {first_name} {last_name}.
They hold a position as a {job_title} at {company}.
Their educational background includes a degree from {university}.
Key skills to include are: {languages}, {frameworks}, and {cloud_platforms}.
The response must be in Markdown format, containing only the resume content and nothing else.
It should include sections for Contact Information, Professional Summary, Education, Technical Skills, Professional Experience, and AI/ML Projects.
Fictional but professional details (names, email, phone, links) should be used.
"""

for i in range(14, 101):
    # Randomly select a new set of data for each resume
    random_first = random.choice(first_names)
    random_last = random.choice(last_names)
    random_title = random.choice(job_titles)
    random_company = random.choice(companies)
    random_university = random.choice(universities)
    random_languages = ', '.join(random.sample(languages, 3))
    random_frameworks = ', '.join(random.sample(frameworks, 2))
    random_cloud = random.choice(cloud_platforms)
    
    # Construct the unique prompt
    prompt = prompt_template.format(
        first_name=random_first,
        last_name=random_last,
        job_title=random_title,
        company=random_company,
        university=random_university,
        languages=random_languages,
        frameworks=random_frameworks,
        cloud_platforms=random_cloud
    )
    
    response = model.generate_content(prompt, generation_config=generation_config)
    
    answer = response.candidates[0].content.parts[0].text
    
    # Convert markdown to HTML
    html_text = markdown.markdown(answer)
    
    # Convert HTML to PDF
    HTML(string=html_text).write_pdf(f"Random_Resume/{i}.pdf")

/media/ishan/9c63e104-0a83-464e-8d4f-7e0675ab17ef/Sports Coach/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 50
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 8
}
]